In [158]:
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
from sklearn.cluster import KMeans
from nltk.corpus import stopwords
import re
import os
import json
import nltk

nltk.download("stopwords")

spanish_stopwords = stopwords.words("spanish")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\malos\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [159]:
data_dir = "../data"
processed_dir = os.path.join(data_dir, "processed")


In [160]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def load_tests(base_dir: str):
	data = defaultdict(lambda: {
		"pre": {"code": None, "full": None},
		"post": {"code": None, "full": None},
	})

	for phase in ["Pre", "Post"]:
		phase_key = phase.lower()
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				file_path = os.path.join(phase_dir, file)
				content = load_json(file_path)

				kind = "code" if "code" in file else "full"

				for user_id, user_data in content.items():
					data[user_id][phase_key][kind] = user_data

	return dict(data)


In [161]:
base_dir = os.path.join(processed_dir, "merged")
tests = load_tests(base_dir)


In [162]:
def is_valid_text(text: str):
    if not isinstance(text, str):
        return False

    text = text.strip()
    words = text.split()

    if len(words) < 4:
        return False
    
    if len(set(words)) <= 2:
        return False

    return True


In [163]:
def clean_text(text: str):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Eliminar palabras repetidas
    words = text.split()
    cleaned = []
    for w in words:
        if not cleaned or w != cleaned[-1]:
            cleaned.append(w)

    return " ".join(cleaned)


In [164]:
raw_texts = []
removed_texts = []

for user_id, test in tests.items():
    post_code = test["post"]["code"]
    text = post_code["G04Q07"]

    if is_valid_text(text):
        raw_texts.append(text)
    else:
        removed_texts.append(text)
        
print(removed_texts)


['si', 'si ', 'si', 'no', '', 'si', 'si', 'Sí', '.', 'si', 'si', 'si', 'no mucho', '', 'Si.', 'Si', 'no', 'bueno', 'si', 'si', 'mas o menos', 'si', 'Sí', 'No', 'Si', 'si', 'si', 'si', 'sii', 'si mucho ', 'si', 'Si', 'no mandar fotos', '', '', 'si ', 'si', 'si', 'si', 'si', 'si', 'si', 'No sé', 'nose', 'si', 'si', 'si', 'Sí', '']


In [165]:
texts = [clean_text(text) for text in raw_texts]

for i, text in enumerate(texts, 1):
    print(f"{i}. {text}")


1. que no hay que hablar con personas de internet
2. si he aprendido que no tengo que hablar con personas que no conozco ni enviar fotos mias
3. si a aprender a no hablar con gente por internet desconocida
4. ya sabia todo lo que el juego intenta enseñar
5. tener cuidado con las personas de internet que no conoces y no dar informacion propia o fotos
6. no ya sabia de eso
7. sí he aprendido que no se puede hablar con desconocid s es internet
8. sí a no compartir información con gente que no conozco
9. no hablar por redes
10. ha tener cuidado con los adultos que se hacen pasar por niños
11. esque en la vida real no tengo dos opciones
12. si creo que he aprendicdo
13. si que hay que tener cuidado con lo que haces en redes
14. si a no fiarte de personas desconocidas en internet
15. creo que yo ya lo sabia y que era consciente de todo pero creo que para otras personas les viene bien
16. a ver los peligros de internet
17. si no hablar con randoms
18. he aprendido a que no tengo que hablar co

In [178]:
vectorizer = TfidfVectorizer(
    stop_words=spanish_stopwords,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X = vectorizer.fit_transform(texts)


In [179]:
k = 5
model = KMeans(n_clusters=k)
clusters = model.fit_predict(X)


In [180]:
terms = vectorizer.get_feature_names_out()
print(terms)
print(len(terms))
centroids = model.cluster_centers_

for i in range(k):
    top_words = centroids[i].argsort()[-10:]
    print("\nCluster", i)
    print([terms[j] for j in top_words])


['alguien' 'aprender' 'aprendido' 'aprendido hablar' 'bien' 'conoces'
 'conozco' 'creo' 'cuidado' 'desconocidos' 'enviar' 'enviar fotos' 'fotos'
 'gente' 'hablar' 'hablar personas' 'informacion' 'información' 'internet'
 'personas' 'personas internet' 'redes' 'sabia' 'si' 'si hablar'
 'si tener' 'tener' 'tener cuidado']
28

Cluster 0
['si hablar', 'gente', 'desconocidos', 'información', 'redes', 'personas', 'aprendido', 'si', 'hablar', 'internet']

Cluster 1
['internet', 'personas', 'personas internet', 'redes', 'si', 'si hablar', 'tener', 'si tener', 'tener cuidado', 'sabia']

Cluster 2
['internet', 'personas internet', 'redes', 'si hablar', 'si tener', 'personas', 'sabia', 'bien', 'si', 'creo']

Cluster 3
['fotos', 'gente', 'conoces', 'redes', 'si', 'informacion', 'si tener', 'cuidado', 'tener', 'tener cuidado']

Cluster 4
['si', 'hablar', 'personas', 'aprendido', 'conozco', 'aprendido hablar', 'hablar personas', 'fotos', 'enviar', 'enviar fotos']
